# Interactive Entry Labeling

Label desirable bullish entry candles inside deterministic rolling windows. The planned window controls which candles are written and where **Next** navigates; Plotly zooming and panning never change labeling coverage or progress.

The notebook intentionally excludes the locked test period. Set `LABELING_END_DATE` to the inclusive end of validation before creating a session. Forward outcomes are only calculated when the complete configured horizon also remains before that boundary.

In [ ]:
from __future__ import annotations

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.labeling import (
    LabelingConfig,
    LabelingSession,
    build_labeling_figure,
    create_labeling_session,
    initialize_labeling_tables,
    load_labels,
    load_latest_labeling_session,
    plan_labeling_windows,
    prepare_labeling_frame,
    risk_guide_for_date,
    save_labeling_window,
    slice_chart_context,
    update_risk_guide_traces,
    update_selected_trace,
)

## Session configuration

`WINDOW_SIZE` and `STEP_SIZE` are fixed when a session is created. A typical `80/60` configuration makes three quarters of each next window new while retaining 20 sessions of overlap. The ATR stop multiple and reward/risk ratio remain adjustable in the UI because they are visual calibration aids.

In [ ]:
PROVIDER = "yfinance"
TICKERS = ("ABB.ST",)  # Replace with the ordered labeling universe.
LABEL_FAMILY = "trend_continuation"
LABELING_START_DATE: str | None = None
LABELING_END_DATE: str | None = None  # Required for a new session: validation end, never test end.

NEW_SESSION_CONFIG = LabelingConfig(
    window_size=80,
    step_size=60,
    forward_horizon=10,
    atr_length=14,
    atr_stop_multiple=1.0,
    reward_risk_ratio=2.0,
    commission_rate=0.0025,
    pivot_high_left=10,
    pivot_high_right=10,
    pivot_low_left=10,
    pivot_low_right=10,
    default_heatmap_mode="net_return",
)

In [ ]:
engine = resolve_database_engine()
initialize_labeling_tables(engine)

session = load_latest_labeling_session(
    engine=engine,
    provider=PROVIDER,
    label_family=LABEL_FAMILY,
)
if session is None:
    if LABELING_END_DATE is None:
        raise ValueError(
            "Set LABELING_END_DATE to the inclusive validation end before creating a session."
        )
    session = create_labeling_session(
        engine=engine,
        provider=PROVIDER,
        tickers=TICKERS,
        label_family=LABEL_FAMILY,
        labeling_start_date=LABELING_START_DATE,
        labeling_end_date=LABELING_END_DATE,
        config=NEW_SESSION_CONFIG,
    )
else:
    print(
        "Resuming",
        session.labeling_session_id,
        session.current_ticker,
        f"window {session.current_window_position + 1}",
    )

session

## Interactive workflow

- Click a candle to toggle its positive label.
- Unselected candles in the planned window are saved as negatives.
- Candles reached only by panning outside the planned window remain unlabeled.
- Hover a candle to display close-based ATR stop and take-profit guides.
- **Reset labels** restores the state loaded for the current window without writing.
- **Save** commits the current window without moving.
- **Next** saves atomically and advances according to the planned window sequence, regardless of the current Plotly viewport.

In [ ]:
class LabelingNotebookController:
    def __init__(self, *, engine, session: LabelingSession) -> None:
        self.engine = engine
        self.session = session
        self.config = session.config
        self.heatmap_mode = session.config.default_heatmap_mode
        self.frames: dict[str, pd.DataFrame] = {}
        self.windows_by_ticker = {}
        self.current_frame = pd.DataFrame()
        self.context_frame = pd.DataFrame()
        self.current_window = None
        self.loaded_positive_dates: set[pd.Timestamp] = set()
        self.working_positive_dates: set[pd.Timestamp] = set()
        self.figure = None

        self.status = widgets.HTML()
        self.hover_status = widgets.HTML()
        self.figure_output = widgets.Output()
        self.message_output = widgets.Output()
        self.heatmap = widgets.Dropdown(
            options=(
                ("Net return after commission", "net_return"),
                ("ATR units after commission", "atr_units"),
                ("Risk units after commission", "risk_units"),
            ),
            value=self.heatmap_mode,
            description="Heatmap",
        )
        self.stop_multiple = widgets.FloatSlider(
            value=self.config.atr_stop_multiple,
            min=0.25,
            max=5.0,
            step=0.25,
            description="ATR stop",
            readout_format=".2f",
            continuous_update=False,
        )
        self.reward_risk = widgets.FloatSlider(
            value=self.config.reward_risk_ratio,
            min=0.5,
            max=6.0,
            step=0.25,
            description="Reward/risk",
            readout_format=".2f",
            continuous_update=False,
        )
        self.reset_button = widgets.Button(description="Reset labels", button_style="warning")
        self.restore_view_button = widgets.Button(description="Restore view")
        self.save_button = widgets.Button(description="Save", button_style="info")
        self.next_button = widgets.Button(description="Next", button_style="success")

        self.heatmap.observe(self._on_visual_configuration_change, names="value")
        self.stop_multiple.observe(self._on_visual_configuration_change, names="value")
        self.reward_risk.observe(self._on_visual_configuration_change, names="value")
        self.reset_button.on_click(self._on_reset)
        self.restore_view_button.on_click(self._on_restore_view)
        self.save_button.on_click(self._on_save)
        self.next_button.on_click(self._on_next)

        self._load_prices_and_windows()
        self._load_current_window()

    def _load_prices_and_windows(self) -> None:
        rows = load_bronze_daily_prices(
            engine=self.engine,
            provider=self.session.provider,
            tickers=self.session.tickers,
            end_date=self.session.labeling_end_date,
            columns=("open", "high", "low", "close", "volume"),
        )
        for ticker in self.session.tickers:
            ticker_rows = rows.loc[rows["ticker"] == ticker].copy()
            if ticker_rows.empty:
                self.frames[ticker] = pd.DataFrame()
                self.windows_by_ticker[ticker] = ()
                continue
            prices = ticker_rows.set_index("trading_date")[
                ["open", "high", "low", "close", "volume"]
            ].sort_index()
            frame = prepare_labeling_frame(prices, config=self.config)
            self.frames[ticker] = frame
            self.windows_by_ticker[ticker] = plan_labeling_windows(
                frame.index,
                config=self.config,
                labeling_start_date=self.session.labeling_start_date,
                labeling_end_date=self.session.labeling_end_date,
            )

    def _load_current_window(self) -> None:
        if self.session.completed:
            self.status.value = "<b>Labeling session complete.</b>"
            self.next_button.disabled = True
            self.save_button.disabled = True
            return
        ticker = self.session.current_ticker
        windows = self.windows_by_ticker[ticker]
        if not windows:
            raise ValueError(f"No complete labeling windows are available for {ticker}.")
        if self.session.current_window_position >= len(windows):
            raise ValueError(
                f"Stored window position is outside the available windows for {ticker}."
            )

        self.current_frame = self.frames[ticker]
        self.current_window = windows[self.session.current_window_position]
        self.context_frame = slice_chart_context(
            self.current_frame,
            window=self.current_window,
            config=self.config,
        )
        loaded = load_labels(
            engine=self.engine,
            provider=self.session.provider,
            ticker=ticker,
            start_date=self.current_window.start_date,
            end_date=self.current_window.end_date,
        )
        self.loaded_positive_dates = {trading_date for trading_date, label in loaded.items() if label}
        self.working_positive_dates = set(self.loaded_positive_dates)
        self._build_figure()
        self._update_status()

    def _build_figure(self, *, preserve_range=False) -> None:
        previous_range = None
        if preserve_range and self.figure is not None:
            previous_range = tuple(self.figure.layout.xaxis.range or ())
        base = build_labeling_figure(
            self.context_frame,
            window=self.current_window,
            selected_dates=self.working_positive_dates,
            config=self.config,
            heatmap_mode=self.heatmap_mode,
        )
        self.figure = go.FigureWidget(base)
        if len(previous_range or ()) == 2:
            self.figure.layout.xaxis.range = previous_range
        candle_trace = next(trace for trace in self.figure.data if trace.name == "OHLC")
        candle_trace.on_click(self._on_candle_click)
        candle_trace.on_hover(self._on_candle_hover)
        with self.figure_output:
            self.figure_output.clear_output(wait=True)
            display(self.figure)

    def _on_candle_click(self, trace, points, selector) -> None:
        del trace, selector
        if not points.point_inds:
            return
        point_position = points.point_inds[0]
        trading_date = pd.Timestamp(self.context_frame.index[point_position])
        if trading_date not in set(self.current_window.trading_dates):
            return
        if trading_date in self.working_positive_dates:
            self.working_positive_dates.remove(trading_date)
        else:
            self.working_positive_dates.add(trading_date)
        update_selected_trace(
            self.figure,
            self.context_frame,
            self.working_positive_dates,
        )
        self._update_status()

    def _on_candle_hover(self, trace, points, selector) -> None:
        del trace, selector
        if not points.point_inds:
            return
        trading_date = pd.Timestamp(self.context_frame.index[points.point_inds[0]])
        guide = risk_guide_for_date(
            self.context_frame,
            trading_date=trading_date,
            config=self.config,
        )
        update_risk_guide_traces(self.figure, guide)
        if guide is None:
            self.hover_status.value = f"<b>{trading_date.date()}</b>: ATR unavailable"
            return
        self.hover_status.value = (
            f"<b>{trading_date.date()}</b> &nbsp; Close {guide.entry:.2f} &nbsp; "
            f"ATR {guide.atr:.2f} &nbsp; Stop {guide.stop:.2f} &nbsp; "
            f"Take profit {guide.take_profit:.2f}"
        )

    def _on_visual_configuration_change(self, change) -> None:
        del change
        self.heatmap_mode = self.heatmap.value
        self.config = self.config.with_calibration(
            atr_stop_multiple=self.stop_multiple.value,
            reward_risk_ratio=self.reward_risk.value,
        )
        self._build_figure(preserve_range=True)

    def _on_reset(self, button) -> None:
        del button
        self.working_positive_dates = set(self.loaded_positive_dates)
        update_selected_trace(
            self.figure,
            self.context_frame,
            self.working_positive_dates,
        )
        self._update_status()

    def _on_restore_view(self, button) -> None:
        del button
        self.figure.layout.xaxis.range = (
            self.current_window.start_date,
            self.current_window.end_date,
        )

    def _save(self, *, advance: bool) -> None:
        next_ticker_position = None
        next_window_position = None
        completed = None
        if advance:
            ticker_windows = self.windows_by_ticker[self.session.current_ticker]
            if self.session.current_window_position + 1 < len(ticker_windows):
                next_ticker_position = self.session.current_ticker_position
                next_window_position = self.session.current_window_position + 1
                completed = False
            elif self.session.current_ticker_position + 1 < len(self.session.tickers):
                next_ticker_position = self.session.current_ticker_position + 1
                next_window_position = 0
                completed = False
            else:
                next_ticker_position = self.session.current_ticker_position
                next_window_position = self.session.current_window_position
                completed = True

        self.session = save_labeling_window(
            engine=self.engine,
            labeling_session_id=self.session.labeling_session_id,
            ticker=self.session.current_ticker,
            window=self.current_window,
            positive_dates=self.working_positive_dates,
            config=self.config,
            next_ticker_position=next_ticker_position,
            next_window_position=next_window_position,
            completed=completed,
        )
        self.loaded_positive_dates = set(self.working_positive_dates)
        if advance and not self.session.completed:
            self._load_current_window()
        elif self.session.completed:
            self.status.value = "<b>Labeling session complete.</b>"
            self.next_button.disabled = True
            self.save_button.disabled = True
        else:
            self._update_status(prefix="Saved. ")

    def _on_save(self, button) -> None:
        del button
        with self.message_output:
            self.message_output.clear_output(wait=True)
            try:
                self._save(advance=False)
            except Exception as error:
                print(f"Save failed: {error}")

    def _on_next(self, button) -> None:
        del button
        with self.message_output:
            self.message_output.clear_output(wait=True)
            try:
                self._save(advance=True)
            except Exception as error:
                print(f"Advance failed; the current window was not moved: {error}")

    def _update_status(self, *, prefix="") -> None:
        windows = self.windows_by_ticker[self.session.current_ticker]
        changed = self.working_positive_dates != self.loaded_positive_dates
        self.status.value = (
            f"{prefix}<b>{self.session.current_ticker}</b> &nbsp; "
            f"window {self.session.current_window_position + 1}/{len(windows)} &nbsp; "
            f"{self.current_window.start_date.date()}–{self.current_window.end_date.date()} &nbsp; "
            f"positive {len(self.working_positive_dates)}/{len(self.current_window.trading_dates)} &nbsp; "
            f"{'unsaved changes' if changed else 'saved state'}"
        )

    def display(self) -> None:
        controls = widgets.VBox(
            [
                widgets.HBox([self.heatmap, self.stop_multiple, self.reward_risk]),
                widgets.HBox(
                    [
                        self.reset_button,
                        self.restore_view_button,
                        self.save_button,
                        self.next_button,
                    ]
                ),
                self.status,
                self.hover_status,
                self.message_output,
            ]
        )
        display(controls, self.figure_output)


controller = LabelingNotebookController(engine=engine, session=session)
controller.display()

## Persistence semantics

The label table has one authoritative row per `(provider, ticker, trading_date)`. `label_family` is metadata on that row, not part of the key. Saving an overlapping window updates the same rows rather than creating duplicates. Session progress and all labels for a window are committed in one transaction, so a failed **Next** cannot advance past unsaved work.